In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import joblib

In [2]:
RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")

df_meta = pd.read_csv(RAW / "games_detailed_info.csv", low_memory=False)
df_meta = df_meta.rename(columns={"primary":"name","boardgamemechanic":"mechanic","boardgamecategory":"category"})
df_meta = df_meta[["id","name","description"]].drop_duplicates(subset="id").set_index("id")

df_reviews = pd.read_csv(PROCESSED / "reviews_with_sentiment.csv", low_memory=False)

In [3]:
df_meta["description"] = df_meta["description"].fillna("")

tf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3)
desc_tfidf = tf.fit_transform(df_meta["description"].values)
print("TF-IDF shape:", desc_tfidf.shape)
joblib.dump(tf, "../Dataset/Processed/tfidf_vec.joblib")
joblib.dump(desc_tfidf, "../Dataset/Processed/desc_tfidf.npz")

TF-IDF shape: (21631, 20000)


['../Dataset/Processed/desc_tfidf.npz']

In [4]:
mech = pd.read_csv(RAW / "mechanics.csv", low_memory=False).set_index("BGGId")
themes = pd.read_csv(RAW / "themes.csv", low_memory=False).set_index("BGGId")
subc = pd.read_csv(RAW / "subcategories.csv", low_memory=False).set_index("BGGId")

for df in [mech, themes, subc]:
    df.index = df.index.astype(int)

ids = df_meta.index.astype(int)
mech = mech.reindex(ids).fillna(0)
themes = themes.reindex(ids).fillna(0)
subc = subc.reindex(ids).fillna(0)

content_matrix = np.hstack([mech.values, themes.values, subc.values])
content_matrix = normalize(content_matrix, norm='l2', axis=1)
print("Content matrix shape:", content_matrix.shape)
joblib.dump(content_matrix, "../Dataset/Processed/content_matrix.npy")

Content matrix shape: (21631, 384)


['../Dataset/Processed/content_matrix.npy']

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
games_in_reviews = df_reviews["ID"].unique()
print("Games in reviews sample:", len(games_in_reviews))

from collections import defaultdict
game_texts = defaultdict(list)
for _, row in df_reviews.iterrows():
    gid = int(row["ID"])
    if len(game_texts[gid]) < 50:
        game_texts[gid].append(str(row["comment"]))

game_ids = []
embeddings = []
for gid, texts in game_texts.items():
    emb = model.encode(texts, show_progress_bar=False)
    mean_emb = emb.mean(axis=0)
    game_ids.append(gid)
    embeddings.append(mean_emb)

emb_df = pd.DataFrame(embeddings, index=game_ids)
emb_df.index.name = "id"
emb_df.to_parquet("../Dataset/Processed/game_review_embeddings.parquet")
print("Saved per-game embeddings:", emb_df.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\felix\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Games in reviews sample: 20
Saved per-game embeddings: (20, 384)


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import joblib

def topk_cosine_sparse(X, k=50, batch_size=512):
    """
    X: matrix [N, D] (can be dense or sparse)
    Return CSR sparse similarity [N, N] with only top-k per row.
    """
    # make sure we can slice rows efficiently
    if sp.issparse(X):
        X = sp.csr_matrix(X)

    N = X.shape[0]
    rows, cols, data = [], [], []

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        sims = cosine_similarity(X[start:end], X, dense_output=True)  # [B, N]

        # remove self similarity for rows in this batch
        for i in range(end - start):
            sims[i, start + i] = -1.0

        # take top-k columns per row
        idx = np.argpartition(sims, -k, axis=1)[:, -k:]
        vals = np.take_along_axis(sims, idx, axis=1)

        # sort descending within top-k
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        vals = np.take_along_axis(vals, order, axis=1)

        # write sparse entries
        for i in range(end - start):
            r = start + i
            c = idx[i]
            v = vals[i]

            # keep only positive scores (optional but recommended)
            mask = v > 0
            c = c[mask]
            v = v[mask]

            rows.extend([r] * len(c))
            cols.extend(c.tolist())
            data.extend(v.tolist())

    S = sp.csr_matrix((data, (rows, cols)), shape=(N, N))
    return S


# ---------- DESC (TF-IDF) ----------
desc_tfidf = joblib.load("../Dataset/Processed/desc_tfidf.npz")
desc_topk = topk_cosine_sparse(desc_tfidf, k=50, batch_size=512)
joblib.dump(desc_topk, "../Dataset/Processed/desc_topk_csr.joblib")
print("Saved desc_topk_csr.joblib:", desc_topk.shape, "nnz=", desc_topk.nnz)

# ---------- CONTENT (mechanics+themes+subcategories) ----------
# content_matrix is already created in the previous cell
content_topk = topk_cosine_sparse(content_matrix, k=50, batch_size=256)
joblib.dump(content_topk, "../Dataset/Processed/content_topk_csr.joblib")
print("Saved content_topk_csr.joblib:", content_topk.shape, "nnz=", content_topk.nnz)

# ---------- EMBEDDING (SentenceTransformer) ----------
emb = pd.read_parquet("../Dataset/Processed/game_review_embeddings.parquet").values.astype(np.float32)

k = 50
nn = NearestNeighbors(n_neighbors=k+1, metric="cosine", algorithm="auto")
nn.fit(emb)

dist, ind = nn.kneighbors(emb, return_distance=True)
sim = 1.0 - dist  # cosine distance -> cosine similarity

rows, cols, data = [], [], []
N = emb.shape[0]
for i in range(N):
    for j, s in zip(ind[i], sim[i]):
        if j == i:
            continue
        if s <= 0:
            continue
        rows.append(i)
        cols.append(j)
        data.append(float(s))

emb_topk = sp.csr_matrix((data, (rows, cols)), shape=(N, N))
joblib.dump(emb_topk, "../Dataset/Processed/emb_topk_csr.joblib")
print("Saved emb_topk_csr.joblib:", emb_topk.shape, "nnz=", emb_topk.nnz)

print("DONE: Saved TOP-K sparse similarity matrices (no more NxN dense).")

Saved similarity matrices
